# 01 - Data Preprocessing

This notebook loads the raw `.fa` FASTA files and produces a unified, model-ready CSV dataset.

## 1. Overview

| File | Label | Split |
|---|---|---|
| `AMPlify_AMP_train_common.fa` | 1 (AMP) | train |
| `AMPlify_non_AMP_train_balanced.fa` | 0 (non-AMP) | train |
| `AMPlify_AMP_test_common.fa` | 1 (AMP) | test |
| `AMPlify_non_AMP_test_balanced.fa` | 0 (non-AMP) | test |

**Label convention**: `AMP = 1`, `non-AMP = 0`

**result**

| Split | AMP (1) | non-AMP (0) | Total |
|-------|---------|-------------|-------|
| train | 3338    | 3338        | 6676  |
| test  | 835     | 835         | 1670  |
| **Total** | **4173** | **4173** | **8346** |

The dataset is **perfectly balanced** in both splits. Output saved to `data/processed_dataset.csv`.

In [9]:
import pandas as pd
import os
import re
import subprocess
from google.colab import userdata

pat_token = userdata.get('PAT')

if not os.path.exists("/content/CE7076-Final-Group2"):
    !git clone https://{pat_token}@github.com/CCisme99/CE7076-Final-Group2.git /content/CE7076-Final-Group2

os.chdir("/content/CE7076-Final-Group2")
DATA_DIR = "data/"
OUTPUT_CSV = "data/processed_dataset.csv"
print("Ready! Files in data/:")
!ls data/

Ready! Files in data/:
AMPlify_AMP_test_common.fa	    AMPlify_non_AMP_train_balanced.fa
AMPlify_AMP_train_common.fa	    AMPlify_non_AMP_train_imbalanced.fa
AMPlify_non_AMP_test_balanced.fa    processed_dataset.csv
AMPlify_non_AMP_test_imbalanced.fa


## Parse FASTA

In [10]:
def parse_fasta(filepath):

    """Return list of amino-acid sequences from a FASTA file."""
    sequences = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('>'):
                sequences.append(line)
    return sequences

# Quick sanity check
sample = parse_fasta(os.path.join(DATA_DIR, 'AMPlify_AMP_train_common.fa'))
print(f'First sequence: {sample[0]}')
print(f'Total AMP train sequences: {len(sample)}')

First sequence: GLLDTFKNLALNAAKSAGVSVLNSLSCKLSKTC
Total AMP train sequences: 3338


## Load and combine files

In [11]:
file_config = [
    ('AMPlify_AMP_train_common.fa',       1, 'train'),
    ('AMPlify_non_AMP_train_balanced.fa', 0, 'train'),
    ('AMPlify_AMP_test_common.fa',        1, 'test'),
    ('AMPlify_non_AMP_test_balanced.fa',  0, 'test'),
]

records = []
for filename, label, split in file_config:
    seqs = parse_fasta(os.path.join(DATA_DIR, filename))
    for seq in seqs:
        records.append({'sequence': seq, 'label': label, 'set': split})
    print(f'{filename}: {len(seqs)} sequences  (label={label}, set={split})')

df = pd.DataFrame(records)
print(f'\nTotal rows: {len(df)}')

AMPlify_AMP_train_common.fa: 3338 sequences  (label=1, set=train)
AMPlify_non_AMP_train_balanced.fa: 3338 sequences  (label=0, set=train)
AMPlify_AMP_test_common.fa: 835 sequences  (label=1, set=test)
AMPlify_non_AMP_test_balanced.fa: 835 sequences  (label=0, set=test)

Total rows: 8346


## Data exploaration

In [12]:
print('=== Shape ===')
print(df.shape)

print('\n=== Sample rows ===')
print(df.head(6).to_string(index=False))

print('\n=== Class distribution ===')
print(df.groupby(['set', 'label']).size().rename('count').reset_index())

=== Shape ===
(8346, 3)

=== Sample rows ===
                                                                            sequence  label   set
                                                   GLLDTFKNLALNAAKSAGVSVLNSLSCKLSKTC      1 train
                                                                   AKKPVAKKAAGGVKKPK      1 train
                                                                 GIIDIAKKLVGGIRNVLGI      1 train
                               MAGFLKVVQLLAKYGSKAVQWAWANKGKILDWLNAGQAIDWVVSKIKQILGIK      1 train
YGPGDGHGGGHGGGHGGGHGNGQGGGHGHGPGGGFGGGHGGGHGGGGRGGGGSGGGGSPGHGAGGGYPGGHGGGHHGGYQTHGY      1 train
                                                        ALFSILRGLKKLGNMGQAFVNCKIYKKC      1 train

=== Class distribution ===
     set  label  count
0   test      0    835
1   test      1    835
2  train      0   3338
3  train      1   3338


In [13]:
# Sequence length statistics
df['seq_len'] = df['sequence'].str.len()
print('=== Sequence length statistics by label ===')
print(df.groupby('label')['seq_len'].describe().round(1))

=== Sequence length statistics by label ===
        count  mean   std  min   25%   50%   75%    max
label                                                  
0      4173.0  29.7  19.9  2.0  16.0  27.0  35.0  183.0
1      4173.0  29.8  19.7  2.0  18.0  25.0  35.0  183.0


## Validation

In [14]:
# Check for duplicates
dup_count = df.duplicated(subset='sequence').sum()
print(f'Duplicate sequences: {dup_count}')

# Check for empty sequences
empty_count = (df['sequence'].str.len() == 0).sum()
print(f'Empty sequences:     {empty_count}')

# Check valid amino acid alphabet
valid_aa = re.compile(r'^[ACDEFGHIKLMNPQRSTVWYBZXU]+$', re.IGNORECASE)
invalid = df[~df['sequence'].str.match(valid_aa)]
print(f'Invalid AA sequences: {len(invalid)}')

if dup_count == 0 and empty_count == 0 and len(invalid) == 0:
    print('\nAll checks passed!')

Duplicate sequences: 0
Empty sequences:     0
Invalid AA sequences: 0

All checks passed!


## Save

In [15]:
df_out = df[['sequence', 'label', 'set']]

# 建立資料夾並儲存
os.makedirs('preprocessing', exist_ok=True)
OUTPUT_CSV = 'preprocessing/processed_dataset.csv'
df_out.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(df_out)} rows to: {OUTPUT_CSV}')
print('\nPreview:')
print(df_out.head(4).to_string(index=False))

Saved 8346 rows to: preprocessing/processed_dataset.csv

Preview:
                                             sequence  label   set
                    GLLDTFKNLALNAAKSAGVSVLNSLSCKLSKTC      1 train
                                    AKKPVAKKAAGGVKKPK      1 train
                                  GIIDIAKKLVGGIRNVLGI      1 train
MAGFLKVVQLLAKYGSKAVQWAWANKGKILDWLNAGQAIDWVVSKIKQILGIK      1 train


In [20]:
!git config user.email "iamyuju.h@gmial.com"
!git config user.name "1syuju"
!git remote set-url origin https://{pat_token}@github.com/CCisme99/CE7076-Final-Group2.git
!git add {OUTPUT_CSV}
!git commit -m "data_prepocessing via Colab"
!git push

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (4/4), 135.35 KiB | 135.35 MiB/s, done.
Total 4 (delta 1), reused 1 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/CCisme99/CE7076-Final-Group2.git
   988c6cc..423d8ef  main -> main
